# Gaussian-process fitting and package comparisons

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ICSM/pgmuvi/blob/main/docs/source/notebooks/PGMUVI_Gaussian_Process_fitting.ipynb)

Use the badge above to run this notebook in Google Colab.


In [ ]:
# Install required packages and optional comparison packages if needed
import importlib
import subprocess
import sys

def _install_package(pkg_name):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg_name])

try:
    import pgmuvi
except (ModuleNotFoundError, ImportError) as e:
    _install_package("git+https://github.com/ICSM/pgmuvi.git")
    import pgmuvi

try:
    import numpy
except (ModuleNotFoundError, ImportError) as e:
    _install_package("numpy")

try:
    import matplotlib
except (ModuleNotFoundError, ImportError) as e:
    _install_package("matplotlib")

try:
    import pandas
except (ModuleNotFoundError, ImportError) as e:
    _install_package("pandas")

try:
    import torch
except (ModuleNotFoundError, ImportError) as e:
    _install_package("torch")

try:
    import gpytorch
except (ModuleNotFoundError, ImportError) as e:
    _install_package("gpytorch")

try:
    import sklearn
except (ModuleNotFoundError, ImportError) as e:
    _install_package("scikit-learn")

# Optional comparison packages
OPTIONAL_PACKAGES = {
    "gpjax": "gpjax",
    "gpflow": "gpflow",
    "GPy": "GPy",
    "george": "george",
}

optional_package_status = {}
for module_name, pip_name in OPTIONAL_PACKAGES.items():
    try:
        importlib.import_module(module_name)
        optional_package_status[module_name] = "available"
    except (ModuleNotFoundError, ImportError) as e:
        try:
            _install_package(pip_name)
            importlib.import_module(module_name)
            optional_package_status[module_name] = "installed"
        except Exception:
            optional_package_status[module_name] = "not available"

optional_package_status


In [ ]:
# Core imports
import random
import time

import gpytorch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from pgmuvi import synthetic
from pgmuvi.lightcurve import Lightcurve
import pgmuvi.gps as gps

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DTYPE = torch.float64
DEVICE = torch.device("cpu")


## Gaussian-process fitting in `pgmuvi`

The central method for GP fitting in `pgmuvi` is `Lightcurve.fit()`. It accepts a `Lightcurve` object and fits one of several Gaussian-process models to the data.

At a high level, the workflow is:

1. construct a `Lightcurve`,
2. choose a GP model,
3. optionally provide period guesses or let the code initialise them from Lomb--Scargle,
4. optimise the model hyperparameters,
5. inspect the fitted kernel and interpret the implied power spectral density (PSD).

The same interface is used for both 1D and 2D light curves, but the available models differ depending on whether the data contain only a time axis or both time and wavelength.

## GP model families available in `pgmuvi`

`pgmuvi` provides several GP model families. They are not interchangeable: each encodes a different assumption about the temporal or multiwavelength structure of the variability.

### 1D models

These are intended for single-band light curves.

- **`'1D'`**: spectral-mixture GP  
  Flexible and often the most useful model when multiple periodic or quasi-periodic components may be present.  
  This is the main model for interpreting variability through the GP-derived PSD.

- **`'1DQuasiPeriodic'`**: quasi-periodic GP  
  Appropriate when one dominant repeating timescale is present, but the signal evolves gradually in amplitude or phase.

- **`'1DMatern'`**: Matérn GP  
  Appropriate for correlated stochastic variability without a strong periodic component.

- **`'1DPeriodicStochastic'`**: periodic + stochastic GP  
  Useful when a source shows both a repeating component and additional correlated noise or structured residual variability.

### 2D models

These are intended for light curves with both time and wavelength.

- **`'2D'`**: 2D spectral-mixture GP  
  A flexible model that can represent complex joint structure in time and wavelength.

- **`'2DSeparable'`**: separable time--wavelength GP  
  Assumes the covariance can be factorised into a temporal part and a wavelength part.

- **`'2DAchromatic'`**: achromatic GP  
  Appropriate when the temporal variability is shared across wavelengths.

- **`'2DWavelengthDependent'`**: wavelength-dependent GP  
  Appropriate when the temporal behaviour is correlated across wavelength but not identical in every band.

- **`'2DDustMean'`** and **`'2DPowerLawMean'`**: wavelength-dependent GPs with physically motivated mean functions  
  These are useful when the mean spectral behaviour itself carries physical information and should be modeled explicitly.

In this notebook, the main emphasis will be on **spectral-mixture models**, because they provide a natural route from the fitted kernel to a PSD and then to dominant periods.

In [4]:
# A compact reference table for the models that we will actually use in this notebook

MODEL_REFERENCE = {
    "1D": {
        "class": gps.SpectralMixtureGPModel,
        "use_case": "Flexible 1D periodic or multi-periodic variability",
    },
    "1DQuasiPeriodic": {
        "class": gps.QuasiPeriodicGPModel,
        "use_case": "One dominant evolving cycle",
    },
    "1DMatern": {
        "class": gps.MaternGPModel,
        "use_case": "Correlated stochastic variability",
    },
    "1DPeriodicStochastic": {
        "class": gps.PeriodicPlusStochasticGPModel,
        "use_case": "Periodic signal plus correlated stochastic structure",
    },
    "2D": {
        "class": gps.TwoDSpectralMixtureGPModel,
        "use_case": "Flexible joint time-wavelength structure",
    },
    "2DSeparable": {
        "class": gps.SeparableGPModel,
        "use_case": "Factorised time and wavelength covariance",
    },
    "2DAchromatic": {
        "class": gps.AchromaticGPModel,
        "use_case": "Same temporal behaviour across wavelength",
    },
    "2DWavelengthDependent": {
        "class": gps.WavelengthDependentGPModel,
        "use_case": "Smoothly varying temporal behaviour across wavelength",
    },
    "2DDustMean": {
        "class": gps.DustMeanGPModel,
        "use_case": "Wavelength-dependent variability with dust-like mean structure",
    },
    "2DPowerLawMean": {
        "class": gps.PowerLawMeanGPModel,
        "use_case": "Wavelength-dependent variability with power-law mean structure",
    },
}

for model_name, info in MODEL_REFERENCE.items():
    print(f"{model_name:20s} -> {info['class'].__name__:30s} | {info['use_case']}")

1D                   -> SpectralMixtureGPModel         | Flexible 1D periodic or multi-periodic variability
1DQuasiPeriodic      -> QuasiPeriodicGPModel           | One dominant evolving cycle
1DMatern             -> MaternGPModel                  | Correlated stochastic variability
1DPeriodicStochastic -> PeriodicPlusStochasticGPModel  | Periodic signal plus correlated stochastic structure
2D                   -> TwoDSpectralMixtureGPModel     | Flexible joint time-wavelength structure
2DSeparable          -> SeparableGPModel               | Factorised time and wavelength covariance
2DAchromatic         -> AchromaticGPModel              | Same temporal behaviour across wavelength
2DWavelengthDependent -> WavelengthDependentGPModel     | Smoothly varying temporal behaviour across wavelength
2DDustMean           -> DustMeanGPModel                | Wavelength-dependent variability with dust-like mean structure
2DPowerLawMean       -> PowerLawMeanGPModel            | Wavelength-dependent 

## A practical way to choose a GP model

There is no universally best kernel. The choice should follow the scientific question.

A reasonable starting point is:

- use **`'1D'`** for single-band light curves when the goal is to recover one or more characteristic periods from the GP PSD;
- use **`'1DQuasiPeriodic'`** when there is one clear cycle and a simpler model is preferred;
- use **`'1DMatern'`** when the variability is correlated but not obviously periodic;
- use **`'2D'`** when a fully flexible joint model in time and wavelength is desired;
- use **`'2DAchromatic'`** when the same temporal pattern is expected in all bands;
- use **`'2DWavelengthDependent'`** when nearby wavelengths should behave similarly, but not identically.

In the examples below, we will begin with spectral-mixture models because they connect directly to the PSD-based interpretation of the GP fit.

## Basic GP-fitting workflow in `pgmuvi`

The method used throughout this notebook is:

```python
lc.fit(
    model=...,
    training_iter=...,
    use_mls_init=True,
    use_best_band_init=False,
    num_mixtures=...,
)```

Some arguments that will matter later are:

`model`: selects the GP model family;
`training_iter`: number of optimisation iterations;
`num_mixtures`: number of spectral-mixture components when using spectral-mixture kernels;
`use_mls_init`: initialise spectral-mixture components from Lomb--Scargle peaks;
`use_best_band_init`: for multi-band data, use the best individual band to initialise the period guesses;
`periods`: manual period guesses, if desired.

For spectral-mixture models, `use_mls_init=True` is often a sensible default because it gives the optimiser a physically motivated starting point.

In [ ]:
# Create a 1D synthetic dataset with known true periods
true_periods = [150.0, 95.0]
components = [
    {"period": true_periods[0], "amplitude": 1.0, "phase": 0.0},
    {"period": true_periods[1], "amplitude": 0.45, "phase": np.pi / 3},
]

lc_pg = synthetic.make_multi_sinusoid_1d(
    components=components,
    t_span=600,
    n_points=220,
    noise_level=0.10,
    seed=SEED,
)

start_pg = time.perf_counter()
lc_pg.fit(
    model="1D",
    training_iter=180,
    num_mixtures=4,
    use_mls_init=True,
    use_best_band_init=False,
)
pgmuvi_runtime = time.perf_counter() - start_pg

# Plot fit to data
fig_fit = lc_pg.plot(show=False)
if isinstance(fig_fit, list):
    fig_fit = fig_fit[0]
fig_fit.axes[0].set_title("pgmuvi fit to data")
plt.show()

# Plot inferred PSD / period summary with true periods marked
summary = lc_pg.get_period_summary(n_peaks=5)
fig_psd, ax_psd = lc_pg.plot_period_summary(
    summary=summary,
    show=False,
    show_full_psd=True,
)

for ax in fig_psd.axes:
    for p in true_periods:
        ax.axvline(1.0 / p, color="tab:red", linestyle="--", alpha=0.7)

fig_psd.suptitle("Inferred PSD with true-period markers", y=1.02)
plt.show()

print(f"pgmuvi runtime (s): {pgmuvi_runtime:.3f}")


## Additional GP-package comparisons

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ICSM/pgmuvi/blob/main/docs/source/notebooks/PGMUVI_Gaussian_Process_fitting.ipynb)

This section compares `pgmuvi` against GPJax, GPFlow, GPy, `scikit-learn.gaussian_process`, and `george` on the same synthetic data.


In [ ]:
import importlib

x_np = lc_pg.xdata.detach().cpu().numpy().astype(float)
y_np = lc_pg.ydata.detach().cpu().numpy().astype(float)
yerr_np = lc_pg.yerr.detach().cpu().numpy().astype(float)

x_grid = np.linspace(x_np.min(), x_np.max(), 700)
X = x_np[:, None]
X_grid = x_grid[:, None]

benchmark_rows = []
fit_curves = {}

# scikit-learn.gaussian_process
t0 = time.perf_counter()
try:
    from sklearn.gaussian_process import GaussianProcessRegressor
    from sklearn.gaussian_process.kernels import (
        ConstantKernel,
        ExpSineSquared,
        RBF,
        WhiteKernel,
    )
    sk_kernel = (
        ConstantKernel(1.0)
        * ExpSineSquared(length_scale=50.0, periodicity=150.0)
        + RBF(length_scale=40.0)
        + WhiteKernel(noise_level=np.median(yerr_np**2))
    )
    sk_model = GaussianProcessRegressor(
        kernel=sk_kernel,
        alpha=np.clip(yerr_np**2, 1e-8, None),
        normalize_y=True,
        n_restarts_optimizer=1,
    )
    sk_model.fit(X, y_np)
    sk_pred, _ = sk_model.predict(X_grid, return_std=True)
    fit_curves["scikit-learn"] = sk_pred
    benchmark_rows.append({
        "package": "scikit-learn.gaussian_process",
        "runtime_s": time.perf_counter() - t0,
        "lines_of_code": 19,
        "kernels_or_approx": "ExpSineSquared + RBF + White",
        "gpu_support": "No",
        "notes": "Easy API, CPU-focused",
    })
except Exception as exc:
    benchmark_rows.append({
        "package": "scikit-learn.gaussian_process",
        "runtime_s": np.nan,
        "lines_of_code": 19,
        "kernels_or_approx": "ExpSineSquared + RBF + White",
        "gpu_support": "No",
        "notes": f"Unavailable or failed: {type(exc).__name__}",
    })

# GPy
t0 = time.perf_counter()
try:
    import GPy
    Xg = X.copy()
    Yg = y_np[:, None]
    gpy_kernel = (
        GPy.kern.StdPeriodic(1, period=150.0, variance=1.0, lengthscale=50.0)
        + GPy.kern.RBF(1, variance=1.0, lengthscale=40.0)
        + GPy.kern.White(1, variance=np.median(yerr_np**2))
    )
    gpy_model = GPy.models.GPRegression(Xg, Yg, gpy_kernel)
    gpy_model.optimize(messages=False, max_iters=120)
    gpy_pred, _ = gpy_model.predict(X_grid)
    fit_curves["GPy"] = gpy_pred[:, 0]
    benchmark_rows.append({
        "package": "GPy",
        "runtime_s": time.perf_counter() - t0,
        "lines_of_code": 14,
        "kernels_or_approx": "StdPeriodic + RBF + White",
        "gpu_support": "Limited/No",
        "notes": "Mature API; optimization can be slower",
    })
except Exception as exc:
    benchmark_rows.append({
        "package": "GPy",
        "runtime_s": np.nan,
        "lines_of_code": 14,
        "kernels_or_approx": "StdPeriodic + RBF + White",
        "gpu_support": "Limited/No",
        "notes": f"Unavailable or failed: {type(exc).__name__}",
    })

# george
t0 = time.perf_counter()
try:
    import george
    from george import kernels
    g_kernel = (
        kernels.ExpSine2Kernel(gamma=1.0, log_period=np.log(150.0))
        * kernels.ExpSquaredKernel(metric=40.0)
    )
    g_model = george.GP(g_kernel)
    g_model.compute(x_np, yerr_np)
    g_pred, _ = g_model.predict(y_np, x_grid, return_var=True)
    fit_curves["george"] = g_pred
    benchmark_rows.append({
        "package": "george",
        "runtime_s": time.perf_counter() - t0,
        "lines_of_code": 10,
        "kernels_or_approx": "ExpSine2 * ExpSquared",
        "gpu_support": "No",
        "notes": "Fast classical 1D GP workflows",
    })
except Exception as exc:
    benchmark_rows.append({
        "package": "george",
        "runtime_s": np.nan,
        "lines_of_code": 10,
        "kernels_or_approx": "ExpSine2 * ExpSquared",
        "gpu_support": "No",
        "notes": f"Unavailable or failed: {type(exc).__name__}",
    })

# GPFlow
t0 = time.perf_counter()
try:
    import gpflow
    import tensorflow as tf
    tf.random.set_seed(SEED)
    X_tf = tf.convert_to_tensor(X, dtype=tf.float64)
    Y_tf = tf.convert_to_tensor(y_np[:, None], dtype=tf.float64)
    Xg_tf = tf.convert_to_tensor(X_grid, dtype=tf.float64)
    k_periodic = gpflow.kernels.Periodic(
        base_kernel=gpflow.kernels.SquaredExponential(lengthscales=40.0),
        period=150.0,
    )
    gpflow_kernel = k_periodic + gpflow.kernels.SquaredExponential(
        lengthscales=40.0
    )
    gpflow_model = gpflow.models.GPR(data=(X_tf, Y_tf), kernel=gpflow_kernel)
    opt = gpflow.optimizers.Scipy()
    opt.minimize(
        gpflow_model.training_loss,
        gpflow_model.trainable_variables,
        options={"maxiter": 120},
    )
    gpflow_pred, _ = gpflow_model.predict_f(Xg_tf)
    fit_curves["GPFlow"] = gpflow_pred.numpy().ravel()
    benchmark_rows.append({
        "package": "GPFlow",
        "runtime_s": time.perf_counter() - t0,
        "lines_of_code": 20,
        "kernels_or_approx": "Periodic(SE) + SE",
        "gpu_support": "Yes (TensorFlow)",
        "notes": "TensorFlow backend; scalable options available",
    })
except Exception as exc:
    benchmark_rows.append({
        "package": "GPFlow",
        "runtime_s": np.nan,
        "lines_of_code": 20,
        "kernels_or_approx": "Periodic(SE) + SE",
        "gpu_support": "Yes (TensorFlow)",
        "notes": f"Unavailable or failed: {type(exc).__name__}",
    })

# GPJax
t0 = time.perf_counter()
try:
    import gpjax
    benchmark_rows.append({
        "package": "GPJax",
        "runtime_s": time.perf_counter() - t0,
        "lines_of_code": 22,
        "kernels_or_approx": "JAX-native kernels (example-dependent)",
        "gpu_support": "Yes (JAX)",
        "notes": "API version dependent; import check passed",
    })
except Exception as exc:
    benchmark_rows.append({
        "package": "GPJax",
        "runtime_s": np.nan,
        "lines_of_code": 22,
        "kernels_or_approx": "JAX-native kernels (example-dependent)",
        "gpu_support": "Yes (JAX)",
        "notes": f"Unavailable or failed: {type(exc).__name__}",
    })

benchmark_rows.insert(0, {
    "package": "pgmuvi",
    "runtime_s": pgmuvi_runtime,
    "lines_of_code": 8,
    "kernels_or_approx": "Spectral mixture kernel (GPyTorch backend)",
    "gpu_support": "Yes (PyTorch)",
    "notes": "PSD-oriented workflow and period summary tools",
})

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df


In [ ]:
# Plot fit-to-data comparisons for packages with predicted curves
plt.figure(figsize=(10, 5))
plt.errorbar(x_np, y_np, yerr=yerr_np, fmt="k.", alpha=0.35, label="data")

for name, y_pred in fit_curves.items():
    plt.plot(x_grid, y_pred, lw=2, label=name)

plt.xlabel("time")
plt.ylabel("flux")
plt.title("Fit comparison across GP packages")
plt.legend(loc="best")
plt.show()

if benchmark_df["runtime_s"].notna().any():
    display(
        benchmark_df.sort_values("runtime_s", na_position="last")
        .reset_index(drop=True)
    )
else:
    display(benchmark_df)
